### Решение сложного задания: Добавление связи между моделями `Book` и `Category`, реализация вложенных сериализаторов, и обновление Swagger-документации


### **2. Шаги реализации**

---

#### **Шаг 1: Обновление модели `Book`**

1. Откройте файл `models.py`:
   ```bash
   nano api/models.py
   ```

2. Измените модель `Book`, добавив связь с `Category`:
   ```python
   from django.db import models

   class Category(models.Model):
       name = models.CharField(max_length=100, unique=True, verbose_name="Название категории")
       description = models.TextField(blank=True, verbose_name="Описание категории")

       def __str__(self):
           return self.name


   class Book(models.Model):
       title = models.CharField(max_length=200, verbose_name="Название книги")
       author = models.CharField(max_length=100, verbose_name="Автор")
       published_date = models.DateField(verbose_name="Дата публикации")
       category = models.ForeignKey(Category, on_delete=models.CASCADE, related_name="books", verbose_name="Категория")
       
       def __str__(self):
           return self.title
   ```

3. Выполните миграции:
   ```bash
   python manage.py makemigrations api
   python manage.py migrate
   ```

---

#### **Шаг 2: Обновление сериализаторов**

1. Откройте файл `serializers.py`:
   ```bash
   nano api/serializers.py
   ```

2. Настройте вложенный сериализатор:
   - Добавьте сериализатор для категорий, включающий вложенные книги:
   ```python
   from rest_framework import serializers
   from .models import Book, Category

   class BookSerializer(serializers.ModelSerializer):
       class Meta:
           model = Book
           fields = ['id', 'title', 'author', 'published_date', 'category']


   class CategoryDetailSerializer(serializers.ModelSerializer):
       books = BookSerializer(many=True, read_only=True)

       class Meta:
           model = Category
           fields = ['id', 'name', 'description', 'books']
   ```

   - Используйте существующий `CategorySerializer` для вывода категорий без вложенных данных (например, при работе с книгами).

---

#### **Шаг 3: Обновление представлений**

1. Откройте файл `views.py`:
   ```bash
   nano api/views.py
   ```

2. Добавьте новое представление для детального отображения категории с вложенными книгами:
   ```python
   from rest_framework.generics import RetrieveAPIView
   from .models import Category
   from .serializers import CategoryDetailSerializer

   class CategoryDetailView(RetrieveAPIView):
       queryset = Category.objects.prefetch_related('books')
       serializer_class = CategoryDetailSerializer
   ```

---

#### **Шаг 4: Обновление маршрутов**

1. Откройте файл `urls.py`:
   ```bash
   nano api/urls.py
   ```

2. Добавьте маршрут для детального просмотра категории с вложенными книгами:
   ```python
   from django.urls import path
   from .views import (
       CategoryListCreateAPIView,
       CategoryRetrieveUpdateDestroyAPIView,
       CategoryDetailView
   )

   urlpatterns = [
       # Категории
       path('categories/', CategoryListCreateAPIView.as_view(), name='category-list-create'),
       path('categories/<int:pk>/', CategoryRetrieveUpdateDestroyAPIView.as_view(), name='category-detail'),
       path('categories/<int:pk>/detail/', CategoryDetailView.as_view(), name='category-detail-books'),

       # Книги (уже существующие)
       path('books/', ...),
   ]
   ```

---

#### **Шаг 5: Обновление документации Swagger**

1. Откройте файл `views.py` и добавьте описания к методам:
   ```python
   from drf_spectacular.utils import extend_schema, extend_schema_view

   @extend_schema_view(
       get=extend_schema(
           description="Получение категории с вложенными данными (книги в категории).",
           summary="Детальная категория с книгами"
       )
   )
   class CategoryDetailView(RetrieveAPIView):
       queryset = Category.objects.prefetch_related('books')
       serializer_class = CategoryDetailSerializer
   ```

2. Проверьте, что описание обновилось в Swagger-интерфейсе:  
   - Запустите сервер:
     ```bash
     python manage.py runserver
     ```
   - Перейдите в Swagger:  
     [http://127.0.0.1:8000/api/schema/swagger-ui/](http://127.0.0.1:8000/api/schema/swagger-ui/)

---

### **6. Тестирование**

1. **Эндпоинт получения категории с вложенными книгами**:
   - URL: `GET http://127.0.0.1:8000/api/categories/1/detail/`
   - Пример ответа:
     ```json
     {
         "id": 1,
         "name": "Фантастика",
         "description": "Книги жанра фантастика",
         "books": [
             {
                 "id": 1,
                 "title": "Дюна",
                 "author": "Фрэнк Герберт",
                 "published_date": "1965-08-01",
                 "category": 1
             },
             {
                 "id": 2,
                 "title": "1984",
                 "author": "Джордж Оруэлл",
                 "published_date": "1949-06-08",
                 "category": 1
             }
         ]
     }
     ```

---

### **7. Итог**

- Добавлена связь между моделями `Book` и `Category`.
- Реализован вложенный вывод данных с использованием вложенных сериализаторов.
- Обновлена документация Swagger для нового эндпоинта.  